Imports:

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.parser import NSL_KDD_COLUMNS

plt.rcParams['figure.figsize'] = (12, 5)
sns.set_theme(style='darkgrid')

Load Data:

In [ ]:
df = pd.read_csv('../data/raw/KDDTrain+.csv', header=None, names=NSL_KDD_COLUMNS)
print(f'Shape: {df.shape}')
df.head()

Basic info:

In [ ]:
df.info()

Missing values:

In [ ]:
missing = df.isnull().sum()
print('Missing values:', missing[missing > 0].to_dict() or 'None')

Class Dist:

In [ ]:
df['binary_label'] = df['label'].apply(lambda x: 'normal' if x == 'normal' else 'attack')

fig, axes = plt.subplots(1, 2)

df['binary_label'].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue', 'tomato'])
axes[0].set_title('Normal vs Attack')
axes[0].tick_params(axis='x', rotation=0)

df[df['label'] != 'normal']['label'].value_counts().head(10).plot(kind='bar', ax=axes[1], color='tomato')
axes[1].set_title('Top 10 Attack Types')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

Attack Categories

In [ ]:
DOS   = {'back','land','neptune','pod','smurf','teardrop','apache2','udpstorm','processtable','mailbomb'}
PROBE = {'ipsweep','nmap','portsweep','satan','mscan','saint'}
R2L   = {'ftp_write','guess_passwd','imap','multihop','phf','spy','warezclient','warezmaster',
         'sendmail','named','snmpgetattack','snmpguess','worm','xlock','xsnoop','httptunnel'}
U2R   = {'buffer_overflow','loadmodule','perl','rootkit','ps','sqlattack','xterm'}

def categorise(label):
    if label == 'normal': return 'Normal'
    if label in DOS:      return 'DoS'
    if label in PROBE:    return 'Probe'
    if label in R2L:      return 'R2L'
    if label in U2R:      return 'U2R'
    return 'Other'

df['category'] = df['label'].apply(categorise)
df['category'].value_counts().plot(kind='bar')
plt.title('Records by Attack Category')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

Protocol and service 

In [ ]:
fig, axes = plt.subplots(1, 2)

df['protocol_type'].value_counts().plot(kind='bar', ax=axes[0])
axes[0].set_title('Protocol Type')
axes[0].tick_params(axis='x', rotation=0)

df['service'].value_counts().head(15).plot(kind='bar', ax=axes[1])
axes[1].set_title('Top 15 Services')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

Numeric dist

In [ ]:
numeric_cols = ['duration', 'src_bytes', 'dst_bytes', 'count',
                'srv_count', 'serror_rate', 'same_srv_rate', 'dst_host_count']

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    df[col].clip(upper=df[col].quantile(0.99)).hist(ax=axes[i], bins=50, color='steelblue', edgecolor='none')
    axes[i].set_title(col)

plt.suptitle('Numeric Feature Distributions (clipped at 99th percentile)', y=1.02)
plt.tight_layout()
plt.show()

Correlation heatmap

In [ ]:
rate_cols = [c for c in df.columns if 'rate' in c or 'count' in c]
corr = df[rate_cols].corr()

plt.figure(figsize=(14, 10))
sns.heatmap(corr, annot=False, cmap='coolwarm', center=0, linewidths=0.3)
plt.title('Correlation — Rate & Count Features')
plt.tight_layout()
plt.show()